# Lesson 02 — Filtering with `WHERE`

In this lesson:
- Use `WHERE` to ask for **only certain rows** of a table
- Combine conditions with `AND` and `OR`
- Use the shortcuts `IN`, `BETWEEN`, and `LIKE`

Time: 25–30 minutes.

By the end you'll be able to ask very specific questions of the data — like *"only the princesses introduced in the 1990s"* or *"only the ones from Arendelle."* This is where SQL starts feeling powerful.

## Quick recap from Lesson 01

Last time you learned the three pieces of every basic query:

- `SELECT` says **which columns** you want
- `FROM` says **which table** to look in
- `LIMIT` caps **how many rows** come back

So far you've been getting *all* the rows of the princesses table (or the first 10, or whatever you chose with `LIMIT`). Today we add a fourth piece — **`WHERE`** — that lets you say *which rows* you actually want.

## Setup

Run this once per Colab session. Same setup as Lesson 01 — sign in with the same email.

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## What `WHERE` does

`WHERE` is a filter. You give it a condition like *"the year is greater than 2000"*, and it keeps only the rows that match — discarding the rest.

The shape of a query with a `WHERE` clause:

```
SELECT [columns]
FROM [table]
WHERE [condition]
```

The `WHERE` line goes **after** `FROM` and **before** `LIMIT` (if you have one).

> Reminder: `%%bigquery` at the top of the cell is just Colab plumbing. The real SQL is `SELECT`, `FROM`, `WHERE`, etc. — the same query works in BigQuery Console or any other SQL tool.

## Equality — exact matches

The simplest condition is asking for an exact match. Use a single `=` (one equals sign — that's important; in SQL, `=` is comparison, not assignment).

Run the cell below to get only the princesses introduced in 2013:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE year = 2013

You should see **2 rows** — Anna and Elsa, both from Frozen (which came out in 2013). All other rows were filtered out by the `WHERE` clause.

For text values, the syntax is the same — but the value goes inside **single quotes**:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE name = 'Ariel'

**1 row** comes back: Ariel.

A couple of things to note:
- **Single quotes** around `'Ariel'` tell SQL it's text, not a column name.
- Text equality is **case-sensitive**. `'ariel'` (lowercase) would return zero rows.

## Comparison operators

Beyond `=`, you have:

| Operator | Means |
|----------|-------|
| `=`      | equal to |
| `!=` or `<>` | not equal to |
| `<`      | less than |
| `<=`     | less than or equal to |
| `>`      | greater than |
| `>=`     | greater than or equal to |

Try this — princesses introduced before 2000:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE year < 2000

You should see 8 rows: every princess from a movie released in 1999 or earlier.

## Combining conditions with `AND`

When you want **multiple conditions, all of which must be true**, use `AND`. Show me princesses from after 2010 *and* whose home is "Arendelle":

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE year > 2010 AND home = 'Arendelle'

**2 rows:** Anna and Elsa. Both conditions are true for them — they were introduced after 2010 *and* their home is Arendelle. No other rows match both.

## Combining conditions with `OR`

When you want **either condition** to be true, use `OR`. Show me princesses named Ariel *or* Belle:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE name = 'Ariel' OR name = 'Belle'

**2 rows.** Either condition triggers a match.

> **Common mistake:** writing `WHERE name = 'Ariel' OR 'Belle'`. That's not valid SQL — every condition needs to be self-contained. The correct form is `name = 'Ariel' OR name = 'Belle'` (the second `name =` is required).

## `NOT` — flipping a condition

Put `NOT` in front of a condition to invert it.

Princesses *not* from 2013:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE NOT year = 2013

(Most people would write this as `WHERE year != 2013` — same result and easier to read. `NOT` is more useful when wrapping more complex expressions.)

## `IN` — shorthand for many `OR`s

When you have a list of values, `IN` is much cleaner than chaining `OR`s. Compare:

```sql
WHERE name = 'Ariel' OR name = 'Belle' OR name = 'Mulan'
```

vs.

```sql
WHERE name IN ('Ariel', 'Belle', 'Mulan')
```

Same result, way nicer to read. Try it:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE name IN ('Ariel', 'Belle', 'Mulan')

## `BETWEEN` — shorthand for ranges

`BETWEEN` is shorthand for *"greater than or equal to X and less than or equal to Y."* It's **inclusive** on both ends.

Princesses from the 1990s:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE year BETWEEN 1990 AND 1999

Same result as writing `WHERE year >= 1990 AND year <= 1999`. Pick whichever you find more readable.

## `LIKE` — text patterns

`LIKE` matches text patterns. Two wildcard characters:

- **`%`** matches any sequence of characters (zero or more)
- **`_`** matches exactly one character

Princesses whose name starts with `M`:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE name LIKE 'M%'

You should see **Mulan, Merida, and Moana** — every name that starts with M.

Read `LIKE 'M%'` as: *"M followed by anything (or nothing)."*

To match a substring anywhere in the value, put `%` on **both** sides. Princesses whose home contains the word "village":

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE home LIKE '%village%'

**1 row** — Pocahontas, whose home is "Powhatan village."

> **Note on case sensitivity:** `LIKE` is case-sensitive in GoogleSQL by default. `'m%'` (lowercase) would not match "Mulan." If you want case-insensitive matching, you can use the `LOWER()` function or BigQuery's case-insensitive cousin operator. We won't dive into that here.

## A quick note on `NULL`

You'll occasionally run into rows where a column has no value at all — that's called `NULL`. It's not zero, not an empty string — it's *missing*.

To filter `NULL` values, you use **`IS NULL`** or **`IS NOT NULL`**, not `=`. (Comparing anything to `NULL` with `=` doesn't work the way you'd expect, because `NULL` doesn't equal anything — not even another `NULL`.)

We won't see `NULL`s in the princesses table, but you'll meet them in Lesson 08 when we work with attraction height limits. For now: just know they exist, and that you compare them with `IS NULL` / `IS NOT NULL`.

## Quick reminders

- You still have **read-only** access. `WHERE` filters what you read; it doesn't change anything. Experiment freely.
- The `%%bigquery` line is still just Colab plumbing. Drop it in any other tool and the SQL works the same.

## Exercises

For each prompt, write your query in the empty code cell and run it. Type from scratch — don't copy-paste.

### Exercise 1

Show me **all columns** of every princess introduced **after 2010**.

In [ ]:
# Your query here

### Exercise 2

Show me the **name** and **movie** columns for every princess **not** from Frozen.

(The movie name is exactly `'Frozen'`.)

In [ ]:
# Your query here

### Exercise 3

Show me **all columns** of princesses whose home is either `'Atlantica'` or `'Arendelle'`.

Use `IN` to write this cleanly.

In [ ]:
# Your query here

### Exercise 4

Show me **all columns** of princesses whose **name starts with the letter A**.

In [ ]:
# Your query here

### Exercise 5 (bonus)

Show me **all columns** of princesses whose **home contains the word `'cottage'`**.

In [ ]:
# Your query here

---

## Solutions

⚠️ **Spoilers below.** Try each yourself first.

### Solution 1

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE year > 2010

### Solution 2

In [ ]:
%%bigquery
SELECT name, movie
FROM disney_lessons.princesses
WHERE movie != 'Frozen'

### Solution 3

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE home IN ('Atlantica', 'Arendelle')

### Solution 4

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE name LIKE 'A%'

### Solution 5

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.princesses
WHERE home LIKE '%cottage%'

## What you've learned

In this lesson you:

- ✅ Used `WHERE` to filter rows
- ✅ Wrote conditions with `=`, `<`, `>`, `!=`
- ✅ Combined conditions with `AND` and `OR`
- ✅ Used `IN` for lists of values
- ✅ Used `BETWEEN` for ranges
- ✅ Used `LIKE` for text patterns

Filtering is one of the two most-used SQL operations (the other is *joining*, which we get to in Lesson 06).

## Up next

You can now ask for very specific rows. But the rows come back in whatever order BigQuery feels like — not sorted by year, name, or anything else. Next we fix that with **`ORDER BY`**, plus learn how to get only **unique** values with `DISTINCT`.

Open `03_sorting.ipynb`.